# MeteoPrep — exploration

Notebook d'exploration de l'étape 2 : météo mensuelle par hôtel.

**Principe :** les champs d'identification (`hotel_code` … `hotel_lat`, `hotel_lon`) viennent de RodPrep.
La météo est calculée au point `(hotel_lat, hotel_lon)` (stations Meteostat les plus proches).

**Années :** si non fournies → **année en cours**. Les mois manquants de l'année en cours
sont complétés par le **même mois de l'année précédente** (jamais d'imputation à 0).


In [1]:
from pathlib import Path
import sys
from datetime import datetime

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "MeteoPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
ROD_OUTPUT = PREPARE / "RodPrep" / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

print("Année en cours :", datetime.utcnow().year)


Année en cours : 2026


## 1. Entrée — identité hôtel depuis RodPrep

Champs retenus : `hotel_code`, `hotel_name`, `hotel_brand`, `hotel_city`, `hotel_lat`, `hotel_lon`.

On rafraîchit l'entrée depuis RodPrep. Par défaut les années cibles = année en cours
(passer `target_years=(2024, 2025, …)` pour l'historique ventes).


In [4]:
from meteo_prep.prep import MeteoPrep, HOTEL_IDENTITY_COLS, READABLE_WEATHER, default_target_years

# Années non fournies → année en cours. Pour jointure ventes historiques, ex. :
# prep = MeteoPrep(INPUT_DIR, OUTPUT_DIR, target_years=(2023, 2024, 2025, 2026))
prep = MeteoPrep(INPUT_DIR, OUTPUT_DIR)

if not (ROD_OUTPUT / "hotel_lookup.parquet").exists():
    raise FileNotFoundError("Exécuter d'abord RodPrep/Explore/explore.ipynb")

hotels_path = prep.fill_input_from_rod(ROD_OUTPUT)
print("Entrée créée depuis RodPrep :", hotels_path)
print("Années cibles :", prep.target_years, "(défaut =", default_target_years(), ")")

hotels = prep.load_input()
print(f"Hôtels : {len(hotels)}")
geo_ok = hotels["hotel_lat"].notna() & hotels["hotel_lon"].notna()
print(f"Avec lat/lon : {geo_ok.sum()} / {len(hotels)}")
hotels[[c for c in HOTEL_IDENTITY_COLS if c in hotels.columns]]


Entrée créée depuis RodPrep : /media/laghmari/ssd-data/dev/hotels/prepare/MeteoPrep/Input/hotels.parquet
Années cibles : (2026,) (défaut = (2026,) )
Hôtels : 7
Avec lat/lon : 7 / 7


,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_lat,hotel_lon
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,Nice,43.689186,7.240512
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,48.591522,7.754599
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,49.006733,2.519843
3,H6188,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,48.833827,2.256274
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,48.885048,2.329923
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,Megève,45.859165,6.619055
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,48.849778,2.282836


## 2. Météo au point hôtel (`hotel_lat`, `hotel_lon`)

`weather_for_hotel` interroge Meteostat au point `(lat, lon)` sur la fenêtre
années cibles + année précédente (pour l'imputation). Résultat indexé par `(annee, mois)`.


In [3]:
enrich_summary = []
for _, hotel in hotels.iterrows():
    info = prep.weather_for_hotel(hotel)
    by_ym = info.get("weather_by_year_month") or {}
    years = sorted({y for (y, _m) in by_ym.keys()}) if by_ym else []
    enrich_summary.append({
        "hotel_code": info["hotel_code"],
        "hotel_name": info["hotel_name"],
        "hotel_lat": info["hotel_lat"],
        "hotel_lon": info["hotel_lon"],
        "source": info["source"],
        "nb_mois_annee": len(by_ym),
        "annees": ",".join(str(y) for y in years),
        "nb_cles_meteo": info["nb_cles_meteo"],
        "warnings": "; ".join(info["warnings"]) if info["warnings"] else "",
    })

enrich_df = pd.DataFrame(enrich_summary)
print(f"Enrichissements : {len(enrich_df)} hôtels")
enrich_df


Enrichissements : 7 hôtels


,hotel_code,hotel_name,hotel_lat,hotel_lon,source,nb_mois_annee,annees,nb_cles_meteo,warnings
0,H2075,Ibis budget Nice Californie,43.689186,7.240512,hotel_coords,19,"2025,2026",456,
1,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,hotel_coords,19,"2025,2026",456,
2,H0815,Ibis Styles Roissy CDG,49.006733,2.519843,hotel_coords,19,"2025,2026",399,
3,H6188,Mercure Paris Boulogne,48.833827,2.256274,hotel_coords,19,"2025,2026",456,
4,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,hotel_coords,19,"2025,2026",456,
5,HB5I0,Novotel Megève Mont-Blanc,45.859165,6.619055,hotel_coords,19,"2025,2026",399,
6,H3546,Novotel Paris Centre Tour Eiffel,48.849778,2.282836,hotel_coords,19,"2025,2026",456,


## 3. Aperçu — premier hôtel avec coordonnées


In [4]:
sample = hotels[hotels["hotel_lat"].notna() & hotels["hotel_lon"].notna()]
if sample.empty:
    sample = hotels
sample_hotel = sample.iloc[0]

info = prep.weather_for_hotel(sample_hotel)
by_ym = info.get("weather_by_year_month") or {}

print(
    f"Hôtel {info['hotel_code']} @ ({info['hotel_lat']}, {info['hotel_lon']}) "
    f"— {len(by_ym)} mois×années (source={info['source']})"
)

preview_rows = []
for (year, month), metrics in sorted(by_ym.items()):
    row = {"annee": year, "mois": month}
    for k in sorted(metrics)[:3]:
        row[k] = metrics[k]
    preview_rows.append(row)
pd.DataFrame(preview_rows).head(18)


Hôtel H2075 @ (43.689186, 7.240512) — 19 mois×années (source=hotel_coords)


,annee,mois,meteo_ensoleillement_min_max,meteo_ensoleillement_min_mean,meteo_ensoleillement_min_min
0,2025,1,60.0,8.455645,0.0
1,2025,2,60.0,9.224702,0.0
2,2025,3,60.0,12.188172,0.0
3,2025,4,60.0,13.725000,0.0
4,2025,5,60.0,19.018817,0.0
5,2025,6,60.0,27.043056,0.0
6,2025,7,60.0,27.596774,0.0
7,2025,8,60.0,24.947581,0.0
8,2025,9,60.0,22.502778,0.0
9,2025,10,60.0,16.432796,0.0


## 4. Renommage lisible

Métriques déjà en `meteo_{libelle}_{stat}` (mean / min / max). Mapping brut → lisible :


In [5]:
print("Mapping métriques :", READABLE_WEATHER)
# Compat profil mensuel (année en cours si année absente des clés aplaties)
monthly_readable = prep._readable_monthly(by_ym)
readable_rows = []
for month, metrics in sorted(monthly_readable.items()):
    for col, val in sorted(metrics.items()):
        readable_rows.append({"mois": month, "colonne": col, "valeur": val})
readable_preview = pd.DataFrame(readable_rows)
print(f"Colonnes lisibles (année préférée) : {readable_preview['colonne'].nunique() if not readable_preview.empty else 0}")
readable_preview.head(18)


Mapping métriques : {'temp': 'temperature_c', 'dwpt': 'point_rosee_c', 'rhum': 'humidite_pct', 'prcp': 'precipitations_mm', 'snow': 'neige_mm', 'wspd': 'vent_kmh', 'pres': 'pression_hpa', 'tsun': 'ensoleillement_min'}
Colonnes lisibles (année préférée) : 24


,mois,colonne,valeur
0,1,meteo_ensoleillement_min_max,60.000000
1,1,meteo_ensoleillement_min_mean,9.317204
2,1,meteo_ensoleillement_min_min,0.000000
3,1,meteo_humidite_pct_max,100.000000
4,1,meteo_humidite_pct_mean,63.106183
5,1,meteo_humidite_pct_min,17.000000
6,1,meteo_neige_mm_max,0.000000
7,1,meteo_neige_mm_mean,0.000000
8,1,meteo_neige_mm_min,0.000000
9,1,meteo_point_rosee_c_max,10.100000


In [6]:
rows = []
for _, hotel in hotels.iterrows():
    rows.extend(prep._rows_for_hotel(hotel))

expanded = pd.DataFrame(rows)
print(f"Grille brute : {expanded.shape[0]} lignes × {expanded.shape[1]} colonnes")
print(f"Hôtels : {expanded['hotel_code'].nunique()} — années : {sorted(expanded['annee'].unique())}")
expanded.sort_values(["hotel_code", "annee", "mois"]).head(12)


Grille brute : 168 lignes × 28 colonnes
Hôtels : 7 — années : [2025, 2026]


,hotel_code,hotel_name,annee,mois,meteo_temperature_c_mean,meteo_temperature_c_min,meteo_temperature_c_max,meteo_point_rosee_c_mean,meteo_point_rosee_c_min,meteo_point_rosee_c_max,meteo_humidite_pct_mean,meteo_humidite_pct_min,meteo_humidite_pct_max,meteo_precipitations_mm_mean,meteo_precipitations_mm_min,meteo_precipitations_mm_max,meteo_neige_mm_mean,meteo_neige_mm_min,meteo_neige_mm_max,meteo_vent_kmh_mean,meteo_vent_kmh_min,meteo_vent_kmh_max,meteo_pression_hpa_mean,meteo_pression_hpa_min,meteo_pression_hpa_max,meteo_ensoleillement_min_mean,meteo_ensoleillement_min_min,meteo_ensoleillement_min_max
96,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,1,4.338978,-4.2,12.8,2.165860,-5.2,10.2,86.268817,59.0,100.0,0.145148,0.0,4.4,0.0,0.0,0.0,12.795968,1.1,36.0,1016.565054,984.9,1043.0,3.615591,0.0,60.0
97,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,2,4.786012,-2.6,15.2,2.297321,-4.5,10.8,84.556548,58.0,100.0,0.071726,0.0,5.8,0.0,0.0,0.0,10.596726,1.4,29.2,1022.739881,1009.4,1041.7,7.114583,0.0,60.0
98,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,3,8.256317,-0.8,20.0,3.816532,-5.1,11.6,75.387097,35.0,100.0,0.029831,0.0,1.6,0.0,0.0,0.0,10.959946,1.1,25.9,1016.223925,997.3,1034.0,11.385753,0.0,60.0
99,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,4,13.056528,3.9,28.3,6.291528,-2.8,12.0,66.373611,22.0,99.0,0.037222,0.0,3.0,0.0,0.0,0.0,10.883472,1.8,26.3,1016.091944,996.4,1029.8,15.797222,0.0,60.0
100,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,5,16.080511,7.3,30.3,7.837500,-1.1,18.3,61.224462,23.0,97.0,0.045161,0.0,4.2,0.0,0.0,0.0,11.871371,1.1,24.5,1017.274194,1007.1,1025.0,14.341398,0.0,60.0
101,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,6,21.652917,9.9,35.5,12.336250,4.4,19.5,59.209722,16.0,95.0,0.062222,0.0,8.1,0.0,0.0,0.0,10.969167,1.8,29.5,1018.316111,1007.7,1029.0,20.022222,0.0,60.0
102,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,7,20.700134,11.2,38.1,12.939651,2.4,19.4,65.196237,19.0,99.0,0.182258,0.0,12.3,0.0,0.0,0.0,11.110349,1.4,24.5,1016.140054,1002.0,1029.2,18.301075,0.0,60.0
103,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,8,21.144086,12.3,36.1,12.172984,4.7,19.8,60.079301,22.0,97.0,0.049731,0.0,4.8,0.0,0.0,0.0,10.214516,1.1,23.8,1016.618414,998.5,1027.9,21.108871,0.0,60.0
104,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,9,15.730417,7.7,28.8,11.139306,4.7,18.9,75.837500,36.0,100.0,0.076250,0.0,3.0,0.0,0.0,0.0,12.468889,2.2,32.8,1017.319028,1004.9,1027.2,11.644444,0.0,60.0
105,H0373,Mercure Paris Montmartre Sacré-Cœur,2025,10,12.450941,6.3,18.2,9.042876,2.1,16.4,80.641129,46.0,100.0,0.047312,0.0,3.4,0.0,0.0,0.0,13.541398,1.1,32.4,1016.667473,984.9,1033.6,9.763441,0.0,59.0


## 6. Imputation — mois manquants ← année précédente

Pour chaque `(hotel_code, annee, mois)` et chaque colonne `meteo_*` :
1. si la valeur est manquante → prendre le **même mois de l'année N-1** (puis N-2, …) ;
2. **jamais** d'imputation à `0`.

Les mois futurs de l'année en cours sont ainsi complétés par l'année dernière.


In [7]:
meteo_cols = [c for c in expanded.columns if c.startswith("meteo_")]
missing_before = int(expanded[meteo_cols].isna().sum().sum()) if meteo_cols else 0

imputed = prep._impute_missing(expanded)
# Sortie finale : années cibles uniquement
imputed_targets = imputed[imputed["annee"].isin(prep.target_years)].copy()
missing_after = int(imputed_targets[meteo_cols].isna().sum().sum()) if meteo_cols else 0

print(f"NaN meteo avant imputation : {missing_before}")
print(f"NaN meteo après imputation (années cibles) : {missing_after}")
print("(les NaN restants = aucune valeur disponible sur les années antérieures non plus)")
imputed_targets.sort_values(["hotel_code", "annee", "mois"]).head(12)


NaN meteo avant imputation : 954
NaN meteo après imputation (années cibles) : 72
(les NaN restants = aucune valeur disponible sur les années antérieures non plus)


,hotel_code,hotel_name,annee,mois,meteo_temperature_c_mean,meteo_temperature_c_min,meteo_temperature_c_max,meteo_point_rosee_c_mean,meteo_point_rosee_c_min,meteo_point_rosee_c_max,meteo_humidite_pct_mean,meteo_humidite_pct_min,meteo_humidite_pct_max,meteo_precipitations_mm_mean,meteo_precipitations_mm_min,meteo_precipitations_mm_max,meteo_neige_mm_mean,meteo_neige_mm_min,meteo_neige_mm_max,meteo_vent_kmh_mean,meteo_vent_kmh_min,meteo_vent_kmh_max,meteo_pression_hpa_mean,meteo_pression_hpa_min,meteo_pression_hpa_max,meteo_ensoleillement_min_mean,meteo_ensoleillement_min_min,meteo_ensoleillement_min_max
108,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,1,4.919892,-5.3,12.6,2.820565,-6.5,8.9,86.686828,59.0,100.0,0.063710,0.0,3.2,0.160221,0.0,7.0,13.743011,1.1,39.2,1006.609409,988.9,1024.6,3.985215,0.0,51.0
109,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,2,8.523810,-1.3,18.4,6.003571,-4.3,12.4,84.879464,51.0,100.0,0.154167,0.0,2.4,0.000000,0.0,0.0,14.770536,1.4,30.6,1005.709077,979.1,1026.1,5.622024,0.0,60.0
110,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,3,9.261290,0.5,18.3,4.848790,-3.6,12.0,75.479839,36.0,100.0,0.055914,0.0,3.7,0.000000,0.0,0.0,11.252688,1.4,34.2,1019.915323,1003.8,1032.6,15.435484,0.0,60.0
111,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,4,13.293611,3.6,25.1,4.753472,-5.3,12.9,60.830556,14.0,99.0,0.011528,0.0,2.3,0.000000,0.0,0.0,11.123472,0.4,25.2,1020.388750,1007.4,1028.9,20.256944,0.0,60.0
112,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,5,16.671102,4.8,32.7,10.703629,-0.4,19.8,71.372312,23.0,99.0,0.140457,0.0,4.7,0.000000,0.0,0.0,9.567339,1.4,27.4,1016.911694,999.6,1030.3,17.143817,0.0,60.0
113,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,6,22.685139,9.6,39.6,12.675278,4.7,20.9,57.445833,17.0,94.0,0.049306,0.0,3.9,0.000000,0.0,0.0,11.141667,1.8,27.7,1017.832083,1004.1,1026.7,23.013889,0.0,60.0
114,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,7,25.780495,14.8,35.7,11.264706,1.3,15.9,44.256966,17.0,90.0,0.000000,0.0,0.0,0.000000,0.0,0.0,11.454799,1.8,20.9,1020.056347,1012.2,1029.4,31.284830,0.0,60.0
115,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,8,21.144086,12.3,36.1,12.172984,4.7,19.8,60.079301,22.0,97.0,0.049731,0.0,4.8,0.000000,0.0,0.0,10.214516,1.1,23.8,1016.618414,998.5,1027.9,21.108871,0.0,60.0
116,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,9,15.730417,7.7,28.8,11.139306,4.7,18.9,75.837500,36.0,100.0,0.076250,0.0,3.0,0.000000,0.0,0.0,12.468889,2.2,32.8,1017.319028,1004.9,1027.2,11.644444,0.0,60.0
117,H0373,Mercure Paris Montmartre Sacré-Cœur,2026,10,12.450941,6.3,18.2,9.042876,2.1,16.4,80.641129,46.0,100.0,0.047312,0.0,3.4,0.000000,0.0,0.0,13.541398,1.1,32.4,1016.667473,984.9,1033.6,9.763441,0.0,59.0


## 7. Persistance `Output/`

`prep.run()` : entrée → météo lat/lon (année×mois) → imputation N←N-1 → filtre années cibles → fichiers.


In [8]:
# Mode step-by-step déjà fait ci-dessus (expanded → imputed_targets).
# prep.run() rejoue le même pipeline et écrit Output/.
meteo_step = imputed_targets.sort_values(
    ["hotel_code", "annee", "mois"], kind="mergesort"
).reset_index(drop=True)

meteo_final_run = prep.run()
print(f"step-by-step : {meteo_step.shape}")
print(f"prep.run()   : {meteo_final_run.shape}")
print(f"Années : {sorted(meteo_final_run['annee'].unique()) if not meteo_final_run.empty else []}")
print(f"Hôtels : {sorted(meteo_final_run['hotel_code'].dropna().unique())}")
display_cols = ["hotel_code", "hotel_name", "annee", "mois"] + [
    c for c in meteo_final_run.columns if c.startswith("meteo_temperature")
]
meteo_final_run[display_cols].head(12)

print("\nFichiers produits :")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" ", path.name, f"({path.stat().st_size} octets)")


step-by-step : (84, 28)
prep.run()   : (84, 28)
Années : [2026]
Hôtels : ['H0373', 'H0815', 'H2075', 'H3546', 'H6188', 'HB5I0', 'HB6A3']

Fichiers produits :
  meteo_monthly.csv (21588 octets)
  meteo_monthly.parquet (26178 octets)


## 8. Comparaison step-by-step vs tout-en-un

`MonthlyWeather.compute_meteo_final(geo, years)` (ou `prep.compute_meteo_final`) produit
la dataframe finale en un seul appel : géolocalisation + liste d'années → grille
année×mois, imputation N←N-1, filtre sur les années cibles.

On compare avec le résultat du parcours pas-à-pas (§2–6) et avec `prep.run()`.


In [9]:
from meteo_prep.weather import MonthlyWeather

# --- Fonction tout-en-un : geo + années → meteo_final ---
years = prep.target_years
geo = hotels  # hotel_lat / hotel_lon (+ ids)

meteo_final = MonthlyWeather.compute_meteo_final(
    geo,
    years=years,
    lat_col="hotel_lat",
    lon_col="hotel_lon",
    id_cols=("hotel_code", "hotel_name"),
)

# Même API via le wrapper hôtel (retire lat/lon du schéma pipeline)
meteo_final_prep = prep.compute_meteo_final(geo=geo, years=years, use_pure_api=True)

print(f"MonthlyWeather.compute_meteo_final : {meteo_final.shape}")
print(f"prep.compute_meteo_final           : {meteo_final_prep.shape}")
print(f"step-by-step (imputed_targets)     : {meteo_step.shape}")
print(f"prep.run()                         : {meteo_final_run.shape}")
print("Colonnes pure API :", list(meteo_final.columns[:8]), "...")
meteo_final.head(12)


MonthlyWeather.compute_meteo_final : (84, 30)
prep.compute_meteo_final           : (84, 28)
step-by-step (imputed_targets)     : (84, 28)
prep.run()                         : (84, 28)
Colonnes pure API : ['hotel_code', 'hotel_name', 'lat', 'lon', 'annee', 'mois', 'meteo_temperature_c_mean', 'meteo_temperature_c_min'] ...


,hotel_code,hotel_name,lat,lon,annee,mois,meteo_temperature_c_mean,meteo_temperature_c_min,meteo_temperature_c_max,meteo_point_rosee_c_mean,meteo_point_rosee_c_min,meteo_point_rosee_c_max,meteo_humidite_pct_mean,meteo_humidite_pct_min,meteo_humidite_pct_max,meteo_precipitations_mm_mean,meteo_precipitations_mm_min,meteo_precipitations_mm_max,meteo_neige_mm_mean,meteo_neige_mm_min,meteo_neige_mm_max,meteo_vent_kmh_mean,meteo_vent_kmh_min,meteo_vent_kmh_max,meteo_pression_hpa_mean,meteo_pression_hpa_min,meteo_pression_hpa_max,meteo_ensoleillement_min_mean,meteo_ensoleillement_min_min,meteo_ensoleillement_min_max
0,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,1,4.919892,-5.3,12.6,2.820565,-6.5,8.9,86.686828,59.0,100.0,0.063710,0.0,3.2,0.160221,0.0,7.0,13.743011,1.1,39.2,1006.609409,988.9,1024.6,3.985215,0.0,51.0
1,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,2,8.523810,-1.3,18.4,6.003571,-4.3,12.4,84.879464,51.0,100.0,0.154167,0.0,2.4,0.000000,0.0,0.0,14.770536,1.4,30.6,1005.709077,979.1,1026.1,5.622024,0.0,60.0
2,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,3,9.261290,0.5,18.3,4.848790,-3.6,12.0,75.479839,36.0,100.0,0.055914,0.0,3.7,0.000000,0.0,0.0,11.252688,1.4,34.2,1019.915323,1003.8,1032.6,15.435484,0.0,60.0
3,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,4,13.293611,3.6,25.1,4.753472,-5.3,12.9,60.830556,14.0,99.0,0.011528,0.0,2.3,0.000000,0.0,0.0,11.123472,0.4,25.2,1020.388750,1007.4,1028.9,20.256944,0.0,60.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,5,16.671102,4.8,32.7,10.703629,-0.4,19.8,71.372312,23.0,99.0,0.140457,0.0,4.7,0.000000,0.0,0.0,9.567339,1.4,27.4,1016.911694,999.6,1030.3,17.143817,0.0,60.0
5,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,6,22.685139,9.6,39.6,12.675278,4.7,20.9,57.445833,17.0,94.0,0.049306,0.0,3.9,0.000000,0.0,0.0,11.141667,1.8,27.7,1017.832083,1004.1,1026.7,23.013889,0.0,60.0
6,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,7,25.780495,14.8,35.7,11.264706,1.3,15.9,44.256966,17.0,90.0,0.000000,0.0,0.0,0.000000,0.0,0.0,11.454799,1.8,20.9,1020.056347,1012.2,1029.4,31.284830,0.0,60.0
7,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,8,21.144086,12.3,36.1,12.172984,4.7,19.8,60.079301,22.0,97.0,0.049731,0.0,4.8,0.000000,0.0,0.0,10.214516,1.1,23.8,1016.618414,998.5,1027.9,21.108871,0.0,60.0
8,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,9,15.730417,7.7,28.8,11.139306,4.7,18.9,75.837500,36.0,100.0,0.076250,0.0,3.0,0.000000,0.0,0.0,12.468889,2.2,32.8,1017.319028,1004.9,1027.2,11.644444,0.0,60.0
9,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,10,12.450941,6.3,18.2,9.042876,2.1,16.4,80.641129,46.0,100.0,0.047312,0.0,3.4,0.000000,0.0,0.0,13.541398,1.1,32.4,1016.667473,984.9,1033.6,9.763441,0.0,59.0


In [10]:
def _align_for_compare(df: pd.DataFrame) -> pd.DataFrame:
    """Clés + colonnes meteo, tri stable, sans lat/lon."""
    keys = [c for c in ("hotel_code", "hotel_name", "annee", "mois") if c in df.columns]
    meteo_cols = sorted(c for c in df.columns if c.startswith("meteo_"))
    out = df[keys + meteo_cols].copy()
    for c in meteo_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.sort_values(keys, kind="mergesort").reset_index(drop=True)


a = _align_for_compare(meteo_step)
b = _align_for_compare(meteo_final_run)
c = _align_for_compare(meteo_final)
d = _align_for_compare(meteo_final_prep)

print("Shapes alignés :")
print(f"  step-by-step : {a.shape}")
print(f"  prep.run()   : {b.shape}")
print(f"  pure API     : {c.shape}")
print(f"  prep wrapper : {d.shape}")

# Identité structurelle (clés)
keys = ["hotel_code", "annee", "mois"]
for label, left, right in [
    ("step vs run", a, b),
    ("step vs pure", a, c),
    ("pure vs prep.compute_meteo_final", c, d),
]:
    same_keys = left[keys].equals(right[keys])
    print(f"\n{label} — mêmes clés (hotel_code, annee, mois) : {same_keys}")
    if not same_keys:
        print("  left  :", left[keys].head(3).to_dict("records"))
        print("  right :", right[keys].head(3).to_dict("records"))
        continue
    meteo_cols = [col for col in left.columns if col.startswith("meteo_")]
    if not meteo_cols:
        print("  (aucune colonne meteo_ à comparer)")
        continue
    # Comparaison numérique tolérante (NaN == NaN)
    left_m = left[meteo_cols]
    right_m = right[meteo_cols]
    equal = left_m.equals(right_m) or (
        ((left_m - right_m).abs().fillna(0) < 1e-9) | (left_m.isna() & right_m.isna())
    ).all().all()
    # equals ne traite pas NaN==NaN ; version manuelle :
    close = True
    diffs = []
    for col in meteo_cols:
        s1, s2 = left[col], right[col]
        both_na = s1.isna() & s2.isna()
        both_num = s1.notna() & s2.notna()
        if not both_na.all() and not (
            both_na | (both_num & ((s1 - s2).abs() < 1e-6))
        ).all():
            close = False
            mask = ~(both_na | (both_num & ((s1 - s2).abs() < 1e-6)))
            n_diff = int(mask.sum())
            diffs.append((col, n_diff))
    print(f"  valeurs meteo_ identiques (tol 1e-6, NaN ok) : {close}")
    if diffs:
        print("  écarts :", diffs[:5], ("..." if len(diffs) > 5 else ""))

print("\nAperçu pure API (températures) :")
temp_cols = [c for c in meteo_final.columns if c.startswith("meteo_temperature")]
meteo_final[["hotel_code", "annee", "mois"] + temp_cols].head(12)


Shapes alignés :
  step-by-step : (84, 28)
  prep.run()   : (84, 28)
  pure API     : (84, 28)
  prep wrapper : (84, 28)

step vs run — mêmes clés (hotel_code, annee, mois) : True
  valeurs meteo_ identiques (tol 1e-6, NaN ok) : True

step vs pure — mêmes clés (hotel_code, annee, mois) : True
  valeurs meteo_ identiques (tol 1e-6, NaN ok) : True

pure vs prep.compute_meteo_final — mêmes clés (hotel_code, annee, mois) : True
  valeurs meteo_ identiques (tol 1e-6, NaN ok) : True

Aperçu pure API (températures) :


,hotel_code,annee,mois,meteo_temperature_c_mean,meteo_temperature_c_min,meteo_temperature_c_max
0,H0373,2026,1,4.919892,-5.3,12.6
1,H0373,2026,2,8.523810,-1.3,18.4
2,H0373,2026,3,9.261290,0.5,18.3
3,H0373,2026,4,13.293611,3.6,25.1
4,H0373,2026,5,16.671102,4.8,32.7
5,H0373,2026,6,22.685139,9.6,39.6
6,H0373,2026,7,25.780495,14.8,35.7
7,H0373,2026,8,21.144086,12.3,36.1
8,H0373,2026,9,15.730417,7.7,28.8
9,H0373,2026,10,12.450941,6.3,18.2


In [12]:
meteo_final

,hotel_code,hotel_name,lat,lon,annee,mois,meteo_temperature_c_mean,meteo_temperature_c_min,meteo_temperature_c_max,meteo_point_rosee_c_mean,meteo_point_rosee_c_min,meteo_point_rosee_c_max,meteo_humidite_pct_mean,meteo_humidite_pct_min,meteo_humidite_pct_max,meteo_precipitations_mm_mean,meteo_precipitations_mm_min,meteo_precipitations_mm_max,meteo_neige_mm_mean,meteo_neige_mm_min,meteo_neige_mm_max,meteo_vent_kmh_mean,meteo_vent_kmh_min,meteo_vent_kmh_max,meteo_pression_hpa_mean,meteo_pression_hpa_min,meteo_pression_hpa_max,meteo_ensoleillement_min_mean,meteo_ensoleillement_min_min,meteo_ensoleillement_min_max
0,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,1,4.919892,-5.3,12.6,2.820565,-6.5,8.9,86.686828,59.0,100.0,0.063710,0.0,3.2,0.160221,0.0,7.0,13.743011,1.1,39.2,1006.609409,988.9,1024.6,3.985215,0.0,51.0
1,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,2,8.523810,-1.3,18.4,6.003571,-4.3,12.4,84.879464,51.0,100.0,0.154167,0.0,2.4,0.000000,0.0,0.0,14.770536,1.4,30.6,1005.709077,979.1,1026.1,5.622024,0.0,60.0
2,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,3,9.261290,0.5,18.3,4.848790,-3.6,12.0,75.479839,36.0,100.0,0.055914,0.0,3.7,0.000000,0.0,0.0,11.252688,1.4,34.2,1019.915323,1003.8,1032.6,15.435484,0.0,60.0
3,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,4,13.293611,3.6,25.1,4.753472,-5.3,12.9,60.830556,14.0,99.0,0.011528,0.0,2.3,0.000000,0.0,0.0,11.123472,0.4,25.2,1020.388750,1007.4,1028.9,20.256944,0.0,60.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2026,5,16.671102,4.8,32.7,10.703629,-0.4,19.8,71.372312,23.0,99.0,0.140457,0.0,4.7,0.000000,0.0,0.0,9.567339,1.4,27.4,1016.911694,999.6,1030.3,17.143817,0.0,60.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,2026,8,19.828091,10.3,31.8,14.673656,7.2,23.6,73.956989,39.0,99.0,0.093952,0.0,8.8,0.000000,0.0,0.0,8.824059,1.1,23.4,1016.182258,1002.9,1026.4,19.869624,0.0,60.0
80,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,2026,9,15.659444,7.5,28.1,12.464861,7.0,19.1,82.411111,51.0,100.0,0.170556,0.0,8.9,0.000000,0.0,0.0,10.733333,1.1,25.6,1017.915417,1007.0,1025.9,9.738889,0.0,60.0
81,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,2026,10,11.109677,3.5,18.4,8.065323,1.3,14.5,82.418011,48.0,100.0,0.104301,0.0,11.5,0.000000,0.0,0.0,13.122446,0.7,34.9,1017.450000,988.7,1034.0,9.120968,0.0,60.0
82,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,2026,11,6.025000,-6.8,19.5,3.543056,-12.9,13.1,85.211111,27.0,100.0,0.096389,0.0,2.6,0.000000,0.0,0.0,9.283889,1.1,24.1,1015.520556,998.5,1030.3,4.538889,0.0,56.0


In [5]:
from meteo_prep.weather import MonthlyWeather

years = [2025,2026]

meteo_final = MonthlyWeather.compute_meteo_final(
    geo = hotels,
    years=years,
    lat_col="hotel_lat",
    lon_col="hotel_lon",
    id_cols=("hotel_code", "hotel_name"),
)

meteo_final.to_excel(OUTPUT_DIR / "hotel_weather_data.xlsx", index=False)


In [6]:
meteo_final

,hotel_code,hotel_name,hotel_lat,hotel_lon,annee,mois,meteo_temperature_c_mean,meteo_temperature_c_min,meteo_temperature_c_max,meteo_point_rosee_c_mean,meteo_point_rosee_c_min,meteo_point_rosee_c_max,meteo_humidite_pct_mean,meteo_humidite_pct_min,meteo_humidite_pct_max,meteo_precipitations_mm_mean,meteo_precipitations_mm_min,meteo_precipitations_mm_max,meteo_neige_mm_mean,meteo_neige_mm_min,meteo_neige_mm_max,meteo_vent_kmh_mean,meteo_vent_kmh_min,meteo_vent_kmh_max,meteo_pression_hpa_mean,meteo_pression_hpa_min,meteo_pression_hpa_max,meteo_ensoleillement_min_mean,meteo_ensoleillement_min_min,meteo_ensoleillement_min_max
0,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2025,1,4.338978,-4.2,12.8,2.165860,-5.2,10.2,86.268817,59.0,100.0,0.145148,0.0,4.4,0.0,0.0,0.0,12.795968,1.1,36.0,1016.565054,984.9,1043.0,3.615591,0.0,60.0
1,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2025,2,4.786012,-2.6,15.2,2.297321,-4.5,10.8,84.556548,58.0,100.0,0.071726,0.0,5.8,0.0,0.0,0.0,10.596726,1.4,29.2,1022.739881,1009.4,1041.7,7.114583,0.0,60.0
2,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2025,3,8.256317,-0.8,20.0,3.816532,-5.1,11.6,75.387097,35.0,100.0,0.029831,0.0,1.6,0.0,0.0,0.0,10.959946,1.1,25.9,1016.223925,997.3,1034.0,11.385753,0.0,60.0
3,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2025,4,13.056528,3.9,28.3,6.291528,-2.8,12.0,66.373611,22.0,99.0,0.037222,0.0,3.0,0.0,0.0,0.0,10.883472,1.8,26.3,1016.091944,996.4,1029.8,15.797222,0.0,60.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,2025,5,16.080511,7.3,30.3,7.837500,-1.1,18.3,61.224462,23.0,97.0,0.045161,0.0,4.2,0.0,0.0,0.0,11.871371,1.1,24.5,1017.274194,1007.1,1025.0,14.341398,0.0,60.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,2026,8,19.828091,10.3,31.8,14.673656,7.2,23.6,73.956989,39.0,99.0,0.093952,0.0,8.8,0.0,0.0,0.0,8.824059,1.1,23.4,1016.182258,1002.9,1026.4,19.869624,0.0,60.0
164,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,2026,9,15.659444,7.5,28.1,12.464861,7.0,19.1,82.411111,51.0,100.0,0.170556,0.0,8.9,0.0,0.0,0.0,10.733333,1.1,25.6,1017.915417,1007.0,1025.9,9.738889,0.0,60.0
165,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,2026,10,11.109677,3.5,18.4,8.065323,1.3,14.5,82.418011,48.0,100.0,0.104301,0.0,11.5,0.0,0.0,0.0,13.122446,0.7,34.9,1017.450000,988.7,1034.0,9.120968,0.0,60.0
166,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,2026,11,6.025000,-6.8,19.5,3.543056,-12.9,13.1,85.211111,27.0,100.0,0.096389,0.0,2.6,0.0,0.0,0.0,9.283889,1.1,24.1,1015.520556,998.5,1030.3,4.538889,0.0,56.0


In [11]:
meteo_final.to_excel("../Output/hotel_weather_data.xlsx", index = False)